# Lab 4 - Build the Audience, Then Buy It
Name: Artem Novikov
Group: KM-43

## Setup

In [48]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [49]:
DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")

programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

print("Data loaded successfully.")

Using Colab cache for faster access to the 'adtech-lab4-tv-audiences' dataset.
Data loaded successfully.


### Task 1.0 Clean the survey file

In [50]:
# 1. valid adult panel members (R and L, 18+)
va = viewer_attributes.copy()
va["respondent_id"] = va["respondent_id"].astype(str).str.strip()

rl_ids = set(va[va["attr_id"].isin([101, 102])]["respondent_id"])
ages = va[va["attr_id"] == 300].copy()
ages["age_num"] = pd.to_numeric(ages["value"], errors="coerce")
adult_ids = set(ages[(ages["respondent_id"].isin(rl_ids)) & (ages["age_num"] >= 18)]["respondent_id"])

# Helper to normalize respondent_id strings (remove whitespace, trailing .0, leading zeros)
def clean_id(s):
    s = s.str.strip()
    s = s.str.replace(r"\.0+$", "", regex=True)
    s = s.str.lstrip("0")
    return s

# 2. clean survey and log issues
df = survey.copy()
raw_rows = len(df)
bad_log = []

# A. repeated headers
is_header = (df["RESPID"].str.strip() == "RESPID") | (df["KEYWORD"].str.strip() == "KEYWORD")
bad_log.append(("Repeated header rows", int(is_header.sum()), "Dropped"))
df = df[~is_header].copy()

# Normalize fields
df["respid_raw"] = df["RESPID"].str.strip()
df["respid"] = clean_id(df["RESPID"])
df["keyword"] = df["KEYWORD"].str.strip().str.upper()
df["value"] = df["VALUE"].str.strip().str.lower()

# Track ID formatting issues (leading zeros or .0 float suffix)
id_reformatted = (df["respid_raw"] != df["respid"])
bad_log.append(("Malformed RESPID format (leading zeros or .0)", int(id_reformatted.sum()), "Normalized to integer string"))

# Track keyword formatting issues (lowercase or extra spaces)
kw_reformatted = (df["KEYWORD"] != df["keyword"])
bad_log.append(("Malformed KEYWORD format (lowercase or spaces)", int(kw_reformatted.sum()), "Stripped and uppercased"))

# unknown/blank keywords
valid_kws = set(survey_layout[survey_layout["RECTYPE"] == "B"]["KEYWORD"].str.strip().str.upper())
is_bad_kw = ~df["keyword"].isin(valid_kws)
bad_log.append(("Unknown or blank KEYWORD", int(is_bad_kw.sum()), "Dropped"))
df = df[~is_bad_kw].copy()

# non-adult or non-panel IDs
is_bad_id = ~df["respid"].isin(adult_ids)
bad_log.append(("Non-adult or unknown RESPID", int(is_bad_id.sum()), "Dropped"))
df = df[~is_bad_id].copy()

# normalize yes values and drop non-yes
yes_vals = {"1", "1.0", "y", "yes", "true"}
val_nonstandard = df["value"].isin({"1.0", "y", "yes", "true"})
bad_log.append(("Non-standard yes VALUE (1.0, y, yes, TRUE)", int(val_nonstandard.sum()), "Normalized to yes"))

is_not_yes = ~df["value"].isin(yes_vals)
bad_log.append(("Non-yes VALUE (0, 0.0, no, N, blank)", int(is_not_yes.sum()), "Dropped"))
df = df[~is_not_yes].copy()

# exact duplicates
is_dup = df.duplicated(subset=["respid", "keyword"], keep="first")
bad_log.append(("Duplicate (RESPID, KEYWORD) rows", int(is_dup.sum()), "Dropped duplicates"))
df = df[~is_dup].copy()

# keep max
pair_map = {"X710101": 1, "X710102": 2, "X710103": 3}
pair_rows = df[df["keyword"].isin(pair_map.keys())].copy()
pair_rows["pair_val"] = pair_rows["keyword"].map(pair_map)

pair_counts_per_person = pair_rows.groupby("respid")["pair_val"].count()
multi_pair_ids = set(pair_counts_per_person[pair_counts_per_person > 1].index)
dropped_lower_pairs = int(pair_counts_per_person[pair_counts_per_person > 1].sum() - len(multi_pair_ids))
bad_log.append(("Conflicting pair counts for same person", dropped_lower_pairs, "Kept higher pair count"))

# Keep only max pair row per person
pair_rows_sorted = pair_rows.sort_values("pair_val", ascending=False)
keep_pair_idx = pair_rows_sorted.drop_duplicates(subset=["respid"], keep="first").index
drop_pair_idx = set(pair_rows.index) - set(keep_pair_idx)
df = df.drop(index=list(drop_pair_idx)).copy()

# impute missing Any flag
has_any = set(df[df["keyword"] == "X710100"]["respid"])
has_other = set(df[df["keyword"] != "X710100"]["respid"])
missing_any_ids = has_other - has_any
bad_log.append(("Buyer missing explicit Any (X710100) row", len(missing_any_ids), "Set buyer = yes"))

if missing_any_ids:
    imputed_rows = pd.DataFrame({
        "respid": list(missing_any_ids),
        "keyword": "X710100",
        "value": "1"
    })
    df_clean = pd.concat([df[["respid", "keyword", "value"]], imputed_rows], ignore_index=True)
else:
    df_clean = df[["respid", "keyword", "value"]].copy()

# build clean panel-level table
panel_survey = pd.DataFrame({"respondent_id": sorted(adult_ids)})
panel_survey["buyer"] = np.where(panel_survey["respondent_id"].isin(set(df_clean["respid"])), "yes", "no")

best_pairs = pair_rows_sorted.drop_duplicates(subset=["respid"], keep="first").set_index("respid")["pair_val"]
panel_survey["pairs"] = panel_survey["respondent_id"].map(best_pairs).fillna(0).astype(int)
panel_survey["pairs_label"] = panel_survey["pairs"].map({0: "0", 1: "1", 2: "2", 3: "3+"})

panel_survey["stride"] = np.where(panel_survey["respondent_id"].isin(set(df_clean[df_clean["keyword"] == "X710201"]["respid"])), "yes", "no")
panel_survey["kestrel"] = np.where(panel_survey["respondent_id"].isin(set(df_clean[df_clean["keyword"] == "X710202"]["respid"])), "yes", "no")
panel_survey["easywalk"] = np.where(panel_survey["respondent_id"].isin(set(df_clean[df_clean["keyword"] == "X710203"]["respid"])), "yes", "no")

# Print required metrics
print("Survey data rows:", raw_rows)
print("Clean yes answers:", len(df_clean))
print("Adult panel members:", len(panel_survey))
print("Non-buyers:", int((panel_survey["buyer"] == "no").sum()))
print("People with two pair counts:", len(multi_pair_ids))

# Table 1: kind of bad record, rows, what you did
bad_df = pd.DataFrame(bad_log, columns=["Kind of bad record", "Rows", "Action taken"])
display(bad_df)

# Table 2: KEYWORD, your count, layout SAMPLE
counts = df_clean["keyword"].value_counts().rename("Your count")
sample_check = survey_layout[survey_layout["RECTYPE"] == "B"][["KEYWORD", "Text", "SAMPLE"]].copy()
sample_check["Layout SAMPLE"] = sample_check["SAMPLE"].astype(int)
sample_check = sample_check.merge(counts, left_on="KEYWORD", right_index=True, how="left").fillna(0)
sample_check["Your count"] = sample_check["Your count"].astype(int)
sample_check = sample_check[["KEYWORD", "Text", "Your count", "Layout SAMPLE"]]
display(sample_check)

# Compute raw vs clean comparison for the written answer
raw_yes = survey[survey["VALUE"].str.strip().str.lower().isin(yes_vals)].copy()
raw_kestrel_rows = int((raw_yes["KEYWORD"].str.strip().str.upper() == "X710202").sum())
raw_buyer_ids = raw_yes["RESPID"].str.strip().nunique()
print(f"Raw Kestrel yes rows: {raw_kestrel_rows} vs Clean: {int((panel_survey['kestrel']=='yes').sum())}")
print(f"Raw unique buyer RESPIDs: {raw_buyer_ids} (share {raw_buyer_ids/len(panel_survey):.3f}) vs Clean: {int((panel_survey['buyer']=='yes').sum())} (share {(panel_survey['buyer']=='yes').mean():.3f})")

Survey data rows: 18349
Clean yes answers: 15635
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113


,Kind of bad record,Rows,Action taken
0,Repeated header rows,2,Dropped
1,Malformed RESPID format (leading zeros or .0),882,Normalized to integer string
2,Malformed KEYWORD format (lowercase or spaces),1104,Stripped and uppercased
3,Unknown or blank KEYWORD,739,Dropped
4,Non-adult or unknown RESPID,423,Dropped
5,"Non-standard yes VALUE (1.0, y, yes, TRUE)",1112,Normalized to yes
6,"Non-yes VALUE (0, 0.0, no, N, blank)",742,Dropped
7,"Duplicate (RESPID, KEYWORD) rows",933,Dropped duplicates
8,Conflicting pair counts for same person,113,Kept higher pair count
9,Buyer missing explicit Any (X710100) row,238,Set buyer = yes


,KEYWORD,Text,Your count,Layout SAMPLE
2,X710100,Any,4779,4779
3,X710101,1 pair,1810,1810
4,X710102,2 pairs,1308,1308
5,X710103,3 or more pairs,1661,1661
7,X710201,STRIDE,1563,1563
8,X710202,KESTREL,354,354
9,X710203,EASYWALK,1050,1050
10,X710298,OTHER BRAND,2739,2739
11,X710299,DON'T KNOW,371,371


Raw Kestrel yes rows: 432 vs Clean: 354
Raw unique buyer RESPIDs: 6231 (share 0.778) vs Clean: 4779 (share 0.597)


**Answer**: The raw survey file had several issues: 2 repeated header rows, 739 unknown or blank keywords, 423 non-adult or invalid respondent IDs, 742 non-yes values (like 0, no, or blank), and 933 duplicate person-keyword pairs. I also found 113 respondents with two different pair counts (kept the higher one), 238 buyers missing an explicit Any row, plus formatting mess like leading zeros, .0 suffixes, and lowercase keywords. Without cleaning, the raw file would have overstated Kestrel buyers by 78 people (432 raw yes rows vs 354 clean buyers, or +22.0%) and inflated the adult buyer share by 18.1 percentage points (0.778 vs 0.597).

### Task 1.1 Four definitions

In [51]:
# Prepare daily weights for adult panel members (R and L aged 18+)
dw = daily_weights.copy()
dw["respondent_id"] = dw["respondent_id"].astype(str).str.strip()
dw_adults = dw[(dw["resp_type"].isin(["R", "L"])) & (dw["respondent_id"].isin(adult_ids))].copy()

# Merge survey flags onto adult panel
survey_flags = panel_survey.set_index("respondent_id")
dw_adults["is_buyer"] = dw_adults["respondent_id"].map(survey_flags["buyer"]) == "yes"
dw_adults["is_heavy"] = dw_adults["respondent_id"].map(survey_flags["pairs"]) >= 3
dw_adults["is_kestrel"] = dw_adults["respondent_id"].map(survey_flags["kestrel"]) == "yes"

# Identify adults aged 25-54 (attr_ids 325, 335, 350)
a25_54_ids = set(va[(va["attr_id"].isin([325, 335, 350])) & (va["respondent_id"].isin(adult_ids))]["respondent_id"])
dw_adults["is_25_54"] = dw_adults["respondent_id"].isin(a25_54_ids)

# Helper: average daily sum of weights across 31 broadcast days (in millions)
def avg_audience_millions(mask):
    daily_sums = dw_adults[mask].groupby("broadcast_date")["weight"].sum()
    return daily_sums.mean() / 1e6

adults_18p_m = avg_audience_millions(pd.Series(True, index=dw_adults.index))
buyers_m = avg_audience_millions(dw_adults["is_buyer"])
heavy_m = avg_audience_millions(dw_adults["is_heavy"])
kestrel_m = avg_audience_millions(dw_adults["is_kestrel"])
adults_25_54_m = avg_audience_millions(dw_adults["is_25_54"])
kestrel_panel_n = int((panel_survey["kestrel"] == "yes").sum())

print(f"Adults 18+ (M): {adults_18p_m:.1f}")
print(f"Shoe buyers (M): {buyers_m:.1f}")
print(f"Heavy buyers, 3+ pairs (M): {heavy_m:.1f}")
print(f"Kestrel buyers (M): {kestrel_m:.1f}")
print(f"Adults 25-54 (M): {adults_25_54_m:.1f}")
print(f"Kestrel buyers in panel (n): {kestrel_panel_n}")

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


### Task 1.2 Who are they?

In [52]:
# Map sex (201 = Men, 202 = Women) and broad age groups for adult panel members
sex_rows = va[va["attr_id"].isin([201, 202])].drop_duplicates("respondent_id").copy()
sex_map = sex_rows.set_index("respondent_id")["attr_id"].map({201: "Men", 202: "Women"})

age_map = ages.drop_duplicates("respondent_id").set_index("respondent_id")["age_num"]

dw_adults["sex"] = dw_adults["respondent_id"].map(sex_map)
dw_adults["age"] = dw_adults["respondent_id"].map(age_map)
dw_adults["age_group"] = pd.cut(dw_adults["age"], bins=[17, 34, 54, 150], labels=["18-34", "35-54", "55+"])
dw_adults["demo_group"] = dw_adults["sex"] + " " + dw_adults["age_group"].astype(str)

groups_order = ["Men 18-34", "Men 35-54", "Men 55+", "Women 18-34", "Women 35-54", "Women 55+"]
audiences = {
    "Adults 18+": pd.Series(True, index=dw_adults.index),
    "Shoe buyers": dw_adults["is_buyer"],
    "Heavy buyers (3+)": dw_adults["is_heavy"],
    "Kestrel buyers": dw_adults["is_kestrel"],
    "Adults 25-54": dw_adults["is_25_54"]
}

# Compute group shares (average of daily shares across 31 broadcast days)
shares_dict = {}
for aud_name, mask in audiences.items():
    sub = dw_adults[mask & dw_adults["demo_group"].isin(groups_order)]
    daily_group = sub.groupby(["broadcast_date", "demo_group"])["weight"].sum().unstack(fill_value=0)
    daily_shares = daily_group.div(daily_group.sum(axis=1), axis=0)
    shares_dict[aud_name] = daily_shares.mean().reindex(groups_order, fill_value=0)

shares_df = pd.DataFrame(shares_dict).round(3)

# Compute index vs Adults 18+ (share / adults_share * 100)
index_df = pd.DataFrame(index=groups_order)
for aud_name in ["Shoe buyers", "Heavy buyers (3+)", "Kestrel buyers", "Adults 25-54"]:
    index_df[aud_name + " Index"] = (shares_df[aud_name] / shares_df["Adults 18+"] * 100).round(0).astype(int)

profile_table = pd.concat([shares_df, index_df], axis=1)
display(profile_table)

,Adults 18+,Shoe buyers,Heavy buyers (3+),Kestrel buyers,Adults 25-54,Shoe buyers Index,Heavy buyers (3+) Index,Kestrel buyers Index,Adults 25-54 Index
Men 18-34,0.144,0.140,0.111,0.118,0.174,97,77,82,121
Men 35-54,0.160,0.158,0.138,0.189,0.326,99,86,118,204
Men 55+,0.185,0.186,0.189,0.126,0.000,101,102,68,0
Women 18-34,0.144,0.140,0.139,0.181,0.174,97,97,126,121
Women 35-54,0.160,0.178,0.226,0.265,0.326,111,141,166,204
Women 55+,0.208,0.197,0.196,0.121,0.000,95,94,58,0


**Answer**: I would hand the planner the heavy athletic-shoe buyers segment (3+ pairs, 49.6M adults) because it fits Kestrel's brief, gives a large enough panel sample for stable program ratings, and captures high-spending groups like women 35-54 (index 141) without the extreme noise of the small Kestrel-only group (354 panelists) or the broad waste of adults 25-54. General athletic-shoe buyers look almost identical to all adults 18+ (indexes between 95 and 111 across every group) because nearly 60% of US adults buy at least one pair a year, so the definition does not separate real prospects from the general public. Since the purchase data was fused onto the panel using only age, sex, and household income, it cannot tell us if a show's audience has stronger brand affinity or buying intent beyond what those three demographics already predict.

### Task 2.1 Programs by audience

In [53]:
# 1. Prepare October 2024 Prime programs and adult viewings
progs = programs.copy()
progs["start_ts"] = pd.to_datetime(progs["start_ts"])
progs["end_ts"] = pd.to_datetime(progs["end_ts"])
progs["duration_min"] = (progs["end_ts"] - progs["start_ts"]).dt.total_seconds() / 60.0

prime_progs = progs[
    (progs["broadcast_date"].astype(str).between("2024-10-01", "2024-10-31")) &
    (progs["daypart"].str.lower() == "prime")
].copy()

vw = viewings.copy()
vw["respondent_id"] = vw["respondent_id"].astype(str).str.strip()
vw = vw[(vw["resp_type"].isin(["R", "L"])) & (vw["respondent_id"].isin(adult_ids))].copy()
vw["start_ts"] = pd.to_datetime(vw["start_ts"])
vw["end_ts"] = pd.to_datetime(vw["end_ts"])

vw["is_heavy"] = vw["respondent_id"].map(survey_flags["pairs"]) >= 3
vw["is_kestrel"] = vw["respondent_id"].map(survey_flags["kestrel"]) == "yes"

daily_univ_adults = dw_adults.groupby("broadcast_date")["weight"].sum()
daily_univ_heavy = dw_adults[dw_adults["is_heavy"]].groupby("broadcast_date")["weight"].sum()
daily_univ_kestrel = dw_adults[dw_adults["is_kestrel"]].groupby("broadcast_date")["weight"].sum()

# Define the 8 target programs in October 2024 Prime
net_map_check = prime_progs[["network_code", "network_name"]].drop_duplicates().set_index("network_name")["network_code"].to_dict()

prog_specs = [
    ("Sunday Night Football (NBC)", net_map_check.get("NBC", "NBC"), "NFL Football"),
    ("Monday Night Football (ABC)", net_map_check.get("ABC", "ABC"), "NFL Football"),
    ("Dancing with the Stars (ABC)", net_map_check.get("ABC", "ABC"), "Dancing with the Stars"),
    ("NCIS (CBS)", net_map_check.get("CBS", "CBS"), "NCIS"),
    ("Hannity (Fox News)", net_map_check.get("Fox News Channel", "FNC"), "Hannity"),
    ("Unsellable Houses (HGTV)", net_map_check.get("HGTV", "HGTV"), "Unsellable Houses"),
    ("Halloween Wars (Food Network)", net_map_check.get("Food Network", "FOOD"), "Halloween Wars"),
    ("Grotesquerie (FX)", net_map_check.get("FX", "FX"), "Grotesquerie")
]

# 2. Helper to price a telecast from rate_card (program rows win over time rows)
def parse_bcast_min(t_str):
    h, m = map(int, str(t_str).split(":"))
    return h * 60 + m

rc_clean = rate_card.copy()
rc_prog = rc_clean[rc_clean["program_name"].notna() & (rc_clean["program_name"].astype(str) != "None")].copy()
rc_time = rc_clean[rc_clean["program_name"].isna() | (rc_clean["program_name"].astype(str) == "None")].copy()
rc_time["start_m"] = rc_time["start_time"].apply(parse_bcast_min)
rc_time["end_m"] = rc_time["end_time"].apply(parse_bcast_min)

def get_spot_cost(row):
    net = row["network_code"]
    pname = row["program_name"]
    # Check program override first
    p_match = rc_prog[(rc_prog["network_code"] == net) & (rc_prog["program_name"] == pname)]
    if not p_match.empty:
        return float(p_match.iloc[0]["cost_30s_usd"])
    # Otherwise match broadcast weekday and start time
    bdate = pd.to_datetime(row["broadcast_date"])
    wday = bdate.weekday()
    st = row["start_ts"]
    b_min = st.hour * 60 + st.minute
    if st.hour < 6:
        b_min += 24 * 60
    t_sub = rc_time[rc_time["network_code"] == net]
    for _, r in t_sub.iterrows():
        wdays = list(r["weekdays"]) if r["weekdays"] is not None else []
        if wday in wdays and r["start_m"] <= b_min < r["end_m"]:
            return float(r["cost_30s_usd"])
    return np.nan

# 3. Compute telecast-level ratings and costs for all telecasts of the 8 programs
results_t2 = []
target_univ_m = daily_univ_heavy.mean() / 1e6
print(f"Target universe (M): {target_univ_m:.1f}")

for label, ncode, pname in prog_specs:
    tels = prime_progs[(prime_progs["network_code"] == ncode) & (prime_progs["program_name"] == pname)].copy()
    vw_net = vw[vw["network_code"] == ncode]

    tel_metrics = []
    for _, t in tels.iterrows():
        t_start, t_end = t["start_ts"], t["end_ts"]
        t_dur = t["duration_min"]
        b_date = t["broadcast_date"]

        # Overlapping viewings on same network
        ov = vw_net[(vw_net["start_ts"] < t_end) & (vw_net["end_ts"] > t_start)].copy()
        ov_start = np.maximum(ov["start_ts"].values, np.datetime64(t_start))
        ov_end = np.minimum(ov["end_ts"].values, np.datetime64(t_end))
        ov_min = (ov_end - ov_start) / np.timedelta64(1, "m")

        w_min = ov["weight"].values * ov_min

        # Average-minute impressions (viewers in an average minute)
        imp_adults = w_min.sum() / t_dur
        imp_heavy = w_min[ov["is_heavy"].values].sum() / t_dur
        imp_kestrel = w_min[ov["is_kestrel"].values].sum() / t_dur

        # Ratings (0-100 scale)
        rtg_adults = imp_adults / daily_univ_adults[b_date] * 100.0
        rtg_heavy = imp_heavy / daily_univ_heavy[b_date] * 100.0
        rtg_kestrel = imp_kestrel / daily_univ_kestrel[b_date] * 100.0

        cost = get_spot_cost(t)

        tel_metrics.append({
            "rtg_adults": rtg_adults,
            "rtg_heavy": rtg_heavy,
            "rtg_kestrel": rtg_kestrel,
            "imp_adults": imp_adults,
            "imp_heavy": imp_heavy,
            "cost": cost
        })

    tdf = pd.DataFrame(tel_metrics)
    mean_rtg_adults = tdf["rtg_adults"].mean()
    mean_rtg_heavy = tdf["rtg_heavy"].mean()
    mean_rtg_kestrel = tdf["rtg_kestrel"].mean()
    mean_imp_adults = tdf["imp_adults"].mean()
    mean_imp_heavy = tdf["imp_heavy"].mean()
    mean_cost = tdf["cost"].mean()

    comp = mean_imp_heavy / mean_imp_adults
    idx_heavy = mean_rtg_heavy / mean_rtg_adults * 100.0
    idx_kestrel = mean_rtg_kestrel / mean_rtg_adults * 100.0
    cpm_adults = mean_cost / mean_imp_adults * 1000.0
    cpm_target = mean_cost / mean_imp_heavy * 1000.0

    results_t2.append({
        "Program": label,
        "Telecasts": len(tdf),
        "Adults Rating": round(mean_rtg_adults, 2),
        "Target Rating": round(mean_rtg_heavy, 2),
        "Target Viewers (k)": round(mean_imp_heavy / 1000.0, 1),
        "Composition": round(comp, 3),
        "Index": round(idx_heavy, 0),
        "Cost :30 ($)": round(mean_cost, 0),
        "Adults CPM ($)": round(cpm_adults, 2),
        "Target CPM ($)": round(cpm_target, 2),
        "Kestrel Index": round(idx_kestrel, 0)
    })

t2_df = pd.DataFrame(results_t2)
t2_df["Index"] = t2_df["Index"].astype(int)
t2_df["Cost :30 ($)"] = t2_df["Cost :30 ($)"].astype(int)
t2_df["Kestrel Index"] = t2_df["Kestrel Index"].astype(int)

display(t2_df.drop(columns=["Telecasts", "Kestrel Index"]))

Target universe (M): 49.6


,Program,Adults Rating,Target Rating,Target Viewers (k),Composition,Index,Cost :30 ($),Adults CPM ($),Target CPM ($)
0,Sunday Night Football (NBC),5.82,5.79,2865.0,0.195,99,487200,33.12,170.05
1,Monday Night Football (ABC),8.93,8.36,4149.1,0.184,94,723600,32.09,174.40
2,Dancing with the Stars (ABC),3.25,3.14,1564.2,0.191,97,158600,19.33,101.39
3,NCIS (CBS),2.21,2.43,1206.3,0.216,110,178800,32.09,148.22
4,Hannity (Fox News),1.09,0.84,416.9,0.152,77,39900,14.50,95.71
5,Unsellable Houses (HGTV),0.50,0.64,317.2,0.249,127,12900,10.12,40.66
6,Halloween Wars (Food Network),0.28,0.33,161.9,0.228,116,9700,13.67,59.90
7,Grotesquerie (FX),0.26,0.25,124.2,0.188,96,7592,11.50,61.11


### Task 2.2 Index, volume or price?

In [54]:
best_idx_prog = t2_df.loc[t2_df["Index"].idxmax(), "Program"]
most_viewers_prog = t2_df.loc[t2_df["Target Viewers (k)"].idxmax(), "Program"]
lowest_cpm_prog = t2_df.loc[t2_df["Target CPM ($)"].idxmin(), "Program"]

print(f"Highest index (program): {best_idx_prog}")
print(f"Most target viewers (program): {most_viewers_prog}")
print(f"Lowest target CPM (program): {lowest_cpm_prog}")

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): Monday Night Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer**: They are not the same program. Unsellable Houses (HGTV) has the highest index (127) and lowest target CPM ($40.66), while Monday Night Football (ABC) brings the largest target volume (4,149.1k viewers vs 317.2k). Just like the shampoo example from Lecture 5, a high index shows audience concentration and cost efficiency rather than total scale. Buying only the highest-index cable show would cap Kestrel's reach at a small niche and pile up frequency, so the plan should mix efficient cable shows with high-reach broadcast programs.

### Task 2.3 Kestrel buyers

In [55]:
display(t2_df[["Program", "Index", "Kestrel Index"]].rename(columns={"Index": "Heavy-buyer Index"}))

,Program,Heavy-buyer Index,Kestrel Index
0,Sunday Night Football (NBC),99,99
1,Monday Night Football (ABC),94,86
2,Dancing with the Stars (ABC),97,83
3,NCIS (CBS),110,57
4,Hannity (Fox News),77,43
5,Unsellable Houses (HGTV),127,77
6,Halloween Wars (Food Network),116,95
7,Grotesquerie (FX),96,130


**Answer**: The Kestrel index jumps around much more (from 43 on Hannity to 130 on Grotesquerie) because there are only 354 Kestrel buyers in the entire panel. On low-rated cable shows, only a few of those panelists watch, which creates high standard error ($SE = \sqrt{p(1-p)/n}$) and random rating swings. I would trust the heavy-buyer index much more since it uses 1,661 panelists, almost five times the sample size, giving much lower sampling noise. You can check index reliability by counting how many actual in-target panelists watched each telecast, calculating standard errors, or checking if the index holds steady week by week across October.

### Task 3.1 Weights

In [56]:
# 1. Clean ACR sessions using Lab 3 rules: Eastern time, 1-minute edit rule
acr = acr_sessions.copy()
acr["start_et"] = pd.to_datetime(acr["start_utc"], utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)
acr["end_et"] = pd.to_datetime(acr["end_utc"], utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)
acr["dur_sec"] = (acr["end_et"] - acr["start_et"]).dt.total_seconds()

# Apply 1-minute edit rule (>= 60 seconds)
acr = acr[acr["dur_sec"] >= 60].copy()

# Assign broadcast date (06:00 to 05:59 -> subtract 6 hours)
acr["broadcast_date"] = (acr["start_et"] - pd.Timedelta(hours=6)).dt.strftime("%Y-%m-%d")

# 2. Match TVs to households (1-to-1 match via hashed_ip)
# Keep hashed_ips that appear on exactly 1 household in hh_attributes
ip_hh_counts = hh_attributes.groupby("hashed_ip")["household_id"].nunique()
unique_hh_ips = set(ip_hh_counts[ip_hh_counts == 1].index)

# Keep tv_ids that have a single unique hashed_ip across sessions
tv_ip_df = acr[["tv_id", "hashed_ip"]].dropna().drop_duplicates()
tv_ip_counts = tv_ip_df.groupby("tv_id")["hashed_ip"].nunique()
single_ip_tvs = set(tv_ip_counts[tv_ip_counts == 1].index)
tv_ip_df = tv_ip_df[tv_ip_df["tv_id"].isin(single_ip_tvs)]

# Also ensure each hashed_ip maps to only 1 TV (1-to-1 TV <-> Household match)
ip_tv_counts = tv_ip_df.groupby("hashed_ip")["tv_id"].nunique()
one_to_one_ips = set(ip_tv_counts[ip_tv_counts == 1].index) & unique_hh_ips

matched_tvs = tv_ip_df[tv_ip_df["hashed_ip"].isin(one_to_one_ips)].merge(
    hh_attributes[["household_id", "hashed_ip", "head_age_band", "income_band"]],
    on="hashed_ip",
    how="inner"
)

# Map each matched TV into the 12 Census universe cells (Under 65 vs 65+, 6 income bands)
matched_tvs["head_age"] = np.where(matched_tvs["head_age_band"] == "65+", "65+", "Under 65")
matched_tvs = matched_tvs.merge(universe, on=["head_age", "income_band"], how="left")
tv_cell_map = matched_tvs.set_index("tv_id")[["head_age", "income_band", "tv_households"]]

print(f"TVs matched to one household: {len(matched_tvs)}")

# 3. Filter ACR sessions to matched TVs and compute daily cell weights
acr_matched = acr[acr["tv_id"].isin(tv_cell_map.index)].copy()
acr_matched = acr_matched.join(tv_cell_map, on="tv_id")

# In-tab TVs per broadcast_date and cell (TVs active on that broadcast day)
intab_tv_days = acr_matched[["broadcast_date", "tv_id", "head_age", "income_band", "tv_households"]].drop_duplicates()
cell_counts = intab_tv_days.groupby(["broadcast_date", "head_age", "income_band", "tv_households"])["tv_id"].count().reset_index(name="intab_tvs")
cell_counts["weight"] = cell_counts["tv_households"] / cell_counts["intab_tvs"]

# Attach daily weight to each in-tab TV-day
intab_weighted = intab_tv_days.merge(
    cell_counts[["broadcast_date", "head_age", "income_band", "weight"]],
    on=["broadcast_date", "head_age", "income_band"],
    how="left"
)

daily_weight_sum_m = intab_weighted.groupby("broadcast_date")["weight"].sum().mean() / 1e6
print(f"Weights sum per day (M): {daily_weight_sum_m:.1f}")

# 4. Evaluate the 8 reference telecasts
ref_merged = reference_ratings.merge(
    programs[["telecast_id", "program_name", "network_code", "network_name", "broadcast_date", "start_ts", "end_ts"]],
    on="telecast_id",
    how="left"
)
ref_merged["start_ts"] = pd.to_datetime(ref_merged["start_ts"])
ref_merged["end_ts"] = pd.to_datetime(ref_merged["end_ts"])
ref_merged["broadcast_date"] = ref_merged["broadcast_date"].astype(str)

t3_rows = []
for _, row in ref_merged.iterrows():
    tid = row["telecast_id"]
    bdate = row["broadcast_date"]
    net = row["network_code"]
    t_start, t_end = row["start_ts"], row["end_ts"]

    # All matched in-tab TVs on that broadcast date
    day_tvs = intab_weighted[intab_weighted["broadcast_date"] == bdate]
    total_intab_n = len(day_tvs)
    total_intab_w = day_tvs["weight"].sum()

    # TVs that watched this telecast (overlapping viewing on same network)
    sess_sub = acr_matched[
        (acr_matched["broadcast_date"] == bdate) &
        (acr_matched["network_code"] == net) &
        (acr_matched["start_et"] < t_end) &
        (acr_matched["end_et"] > t_start)
    ]
    watched_tv_ids = set(sess_sub["tv_id"].unique())

    watched_day_tvs = day_tvs[day_tvs["tv_id"].isin(watched_tv_ids)]
    unweighted_rtg = len(watched_day_tvs) / total_intab_n * 100.0
    weighted_rtg = watched_day_tvs["weight"].sum() / total_intab_w * 100.0
    ref_rtg = float(row["hh_rating"])

    label = f"{row['program_name']} ({row['network_code']})"
    t3_rows.append({
        "Telecast": label,
        "Date": bdate,
        "Unweighted": round(unweighted_rtg, 2),
        "Weighted": round(weighted_rtg, 2),
        "Reference": round(ref_rtg, 2),
        "Gap Unw": abs(unweighted_rtg - ref_rtg),
        "Gap Wtd": abs(weighted_rtg - ref_rtg)
    })

t3_df = pd.DataFrame(t3_rows)
display(t3_df[["Telecast", "Date", "Unweighted", "Weighted", "Reference"]])

mean_gap_unw = t3_df["Gap Unw"].mean()
mean_gap_wtd = t3_df["Gap Wtd"].mean()
share_closed = (mean_gap_unw - mean_gap_wtd) / mean_gap_unw

print(f"Mean gap unweighted (points): {mean_gap_unw:.2f}")
print(f"Mean gap weighted (points): {mean_gap_wtd:.2f}")
print(f"Share of gap closed: {share_closed:.3f}")

TVs matched to one household: 19329
Weights sum per day (M): 125.8


,Telecast,Date,Unweighted,Weighted,Reference
0,MLB Postseason (FOX),2024-10-25,17.20,17.93,25.86
1,NFL Football (NBC),2024-10-27,22.91,24.04,33.14
2,NCIS (CBS),2024-10-14,5.64,5.93,10.56
3,Hannity (FNC),2024-10-16,1.20,1.52,2.30
4,Unsellable Houses (HGTV),2024-10-16,2.16,2.12,3.69
5,Halloween Baking Championship (FOOD),2024-10-14,1.01,0.97,1.63
6,Grotesquerie (FX),2024-10-16,2.19,2.13,3.53
7,The Real Housewives of New York City (BRAVO),2024-10-15,0.77,0.70,1.23


Mean gap unweighted (points): 3.61
Mean gap weighted (points): 3.33
Share of gap closed: 0.078


### Task 3.2 What weighting cannot fix

**Answer**: Broadcast sports, drama, and cable news (MLB Postseason, NFL Football, NCIS, and Hannity) moved up toward the reference because 65+ households are under-represented in smart-TV data and got higher weights, while younger-skewing cable shows (Unsellable Houses, Halloween Baking Championship, Grotesquerie, and RHONY) moved slightly down because under-65 homes were over-represented. Over 92% of the gap is still there because demographic weighting cannot fix structural ACR coverage gaps: missing over-the-air and unconnected TVs, unmeasured secondary sets in multi-TV homes, streaming apps that block ACR, opt-in bias, and IP-matching errors. Weighting by income barely moves these ratings because viewing differences across income bands are small once age is controlled for, and noisy IP matching weakens the link between a TV and its true household income.

### Task 4.1 October predicts October

In [57]:
# 1. Prepare P2+ (all R and L panel members aged 2+) and Heavy buyers (3+ pairs)
p2_ids = set(va[va["attr_id"].isin([101, 102])]["respondent_id"])
dw_p2 = dw[(dw["resp_type"].isin(["R", "L"])) & (dw["respondent_id"].isin(p2_ids))].copy()
dw_p2["bdate_str"] = dw_p2["broadcast_date"].astype(str)
daily_univ_p2 = dw_p2.groupby("bdate_str")["weight"].sum()

dw_adults["bdate_str"] = dw_adults["broadcast_date"].astype(str)
daily_univ_heavy_str = dw_adults[dw_adults["is_heavy"]].groupby("bdate_str")["weight"].sum()

vw_p2 = viewings.copy()
vw_p2["respondent_id"] = vw_p2["respondent_id"].astype(str).str.strip()
vw_p2 = vw_p2[(vw_p2["resp_type"].isin(["R", "L"])) & (vw_p2["respondent_id"].isin(p2_ids))].copy()
vw_p2["start_ts"] = pd.to_datetime(vw_p2["start_ts"])
vw_p2["end_ts"] = pd.to_datetime(vw_p2["end_ts"])
vw_p2["is_heavy"] = vw_p2["respondent_id"].map(survey_flags["pairs"]) >= 3

# 2. Filter Prime telecasts on Big 4 + Fox News across Oct 1-28, 2024
target_nets = {"CBS", "NBC", "ABC", "FOX", "FNC"}
t4_progs = progs[
    (progs["broadcast_date"].astype(str).between("2024-10-01", "2024-10-28")) &
    (progs["daypart"].str.lower() == "prime") &
    (progs["network_code"].isin(target_nets))
].copy()
t4_progs["bdate_str"] = t4_progs["broadcast_date"].astype(str)

# Compute P2+ and Heavy-buyer rating for each telecast
rtg_p2_list = []
rtg_heavy_list = []

vw_by_net = {net: df_net for net, df_net in vw_p2.groupby("network_code")}

for _, t in t4_progs.iterrows():
    net = t["network_code"]
    t_start, t_end = t["start_ts"], t["end_ts"]
    t_dur = t["duration_min"]
    b_date = t["bdate_str"]

    vw_net = vw_by_net.get(net)
    if vw_net is None or vw_net.empty:
        rtg_p2_list.append(0.0)
        rtg_heavy_list.append(0.0)
        continue

    ov = vw_net[(vw_net["start_ts"] < t_end) & (vw_net["end_ts"] > t_start)]
    if ov.empty:
        rtg_p2_list.append(0.0)
        rtg_heavy_list.append(0.0)
        continue

    ov_start = np.maximum(ov["start_ts"].values, np.datetime64(t_start))
    ov_end = np.minimum(ov["end_ts"].values, np.datetime64(t_end))
    ov_min = (ov_end - ov_start) / np.timedelta64(1, "m")

    w_min = ov["weight"].values * ov_min
    imp_p2 = w_min.sum() / t_dur
    imp_heavy = w_min[ov["is_heavy"].values].sum() / t_dur

    rtg_p2_list.append(imp_p2 / daily_univ_p2[b_date] * 100.0)
    rtg_heavy_list.append(imp_heavy / daily_univ_heavy_str[b_date] * 100.0)

t4_progs["rtg_p2"] = rtg_p2_list
t4_progs["rtg_heavy"] = rtg_heavy_list

# 3. Split history (Oct 1-21) vs forecast week (Oct 22-28)
hist_df = t4_progs[t4_progs["bdate_str"].between("2024-10-01", "2024-10-21")].copy()
fore_df = t4_progs[t4_progs["bdate_str"].between("2024-10-22", "2024-10-28")].copy()

hist_means = hist_df.groupby(["network_code", "program_name"], as_index=False).agg(
    fcst_p2=("rtg_p2", "mean"),
    fcst_heavy=("rtg_heavy", "mean")
)

eval_df = fore_df.merge(hist_means, on=["network_code", "program_name"], how="inner")

# Skip telecasts whose actual rating is 0
eval_p2 = eval_df[eval_df["rtg_p2"] > 0].copy()
eval_heavy = eval_df[eval_df["rtg_heavy"] > 0].copy()

eval_p2["ape_p2"] = np.abs(eval_p2["rtg_p2"] - eval_p2["fcst_p2"]) / eval_p2["rtg_p2"] * 100.0
eval_heavy["ape_heavy"] = np.abs(eval_heavy["rtg_heavy"] - eval_heavy["fcst_heavy"]) / eval_heavy["rtg_heavy"] * 100.0

mape_p2 = eval_p2["ape_p2"].mean()
mape_heavy = eval_heavy["ape_heavy"].mean()

biggest_row = eval_p2.loc[eval_p2["ape_p2"].idxmax()]
biggest_label = f"{biggest_row['program_name']} ({biggest_row['network_code']}), {biggest_row['bdate_str']}"

print(f"Telecasts forecast (n): {len(eval_p2)}")
print(f"MAPE, P2+ (%): {mape_p2:.1f}")
print(f"MAPE, heavy buyers (%): {mape_heavy:.1f}")
print(f"Biggest miss, P2+ (program, date): {biggest_label}")

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX), 2024-10-27


### Task 4.2 Why it missed

In [58]:
top5_misses = eval_p2.sort_values("ape_p2", ascending=False).head(5).copy()
top5_misses["Program"] = top5_misses["program_name"] + " (" + top5_misses["network_code"] + ")"
top5_misses["Date"] = top5_misses["bdate_str"]
top5_misses["Forecast"] = top5_misses["fcst_p2"].round(2)
top5_misses["Actual"] = top5_misses["rtg_p2"].round(2)
top5_misses["APE (%)"] = top5_misses["ape_p2"].round(1)

display(top5_misses[["Program", "Date", "Forecast", "Actual", "APE (%)"]])

,Program,Date,Forecast,Actual,APE (%)
80,Krapopolis (FOX),2024-10-27,1.06,0.22,387.8
78,Universal Basic Guys (FOX),2024-10-27,0.68,0.26,162.2
37,Fire Country (CBS),2024-10-27,1.70,0.67,154.8
65,Accused (FOX),2024-10-22,0.95,0.37,154.6
4,What Would You Do? (ABC),2024-10-23,0.44,0.18,138.1


**Answer**: Four of the five biggest misses happened on Sunday, October 27 (Krapopolis, Universal Basic Guys, and Fire Country) and Tuesday, October 22 (Accused), when live NFL and MLB postseason games ran long and shifted the prime-time schedule and lead-in audiences, while low actual ratings (0.18 to 0.67) magnified the percentage error. The heavy-buyer forecast is less accurate than P2+ (38.6% vs 35.8% MAPE) because heavy buyers are a smaller panel slice (1,661 adults vs all P2+ panelists), meaning fewer in-target viewers per telecast and higher sampling variance. A better forecast would add day-of-week and time-slot controls along with lead-in ratings or live sports overrun flags to account for schedule shifts.

### Task 5.1 The plan

In [59]:
# Allocate $1,000,000 across selected Task 2 programs (whole :30 spots)
spots_plan = {
    "Dancing with the Stars (ABC)": 2,
    "NCIS (CBS)": 1,
    "Unsellable Houses (HGTV)": 25,
    "Halloween Wars (Food Network)": 18
}

plan_rows = []
for prog, spots in spots_plan.items():
    row = t2_df[t2_df["Program"] == prog].iloc[0]
    unit_cost = row["Cost :30 ($)"]
    target_viewers = row["Target Viewers (k)"] * 1000.0
    comp = row["Composition"]

    total_cost = spots * unit_cost
    total_target_imps = spots * target_viewers

    plan_rows.append({
        "Program": prog,
        "Spots": spots,
        "Unit Cost ($)": int(unit_cost),
        "Cost ($)": int(total_cost),
        "Target Impressions (M)": round(total_target_imps / 1e6, 2),
        "Composition": comp
    })

plan_df = pd.DataFrame(plan_rows)
display(plan_df[["Program", "Spots", "Cost ($)", "Target Impressions (M)"]])

total_plan_cost = int(plan_df["Cost ($)"].sum())
total_plan_imps_m = plan_df["Target Impressions (M)"].sum()
blended_target_cpm = total_plan_cost / (total_plan_imps_m * 1e6) * 1000.0

# Weighted average composition by spend (share of budget reaching heavy buyers vs non-heavy buyers)
on_target_spend_share = (plan_df["Cost ($)"] * plan_df["Composition"]).sum() / total_plan_cost
overspray_spend_share = (1.0 - on_target_spend_share) * 100.0

print(f"Plan cost ($): {total_plan_cost}")
print(f"Plan target impressions (M): {total_plan_imps_m:.1f}")
print(f"Blended target CPM ($): {blended_target_cpm:.2f}")
print(f"Budget reaching non-heavy buyers (%): {overspray_spend_share:.1f}")

,Program,Spots,Cost ($),Target Impressions (M)
0,Dancing with the Stars (ABC),2,317200,3.13
1,NCIS (CBS),1,178800,1.21
2,Unsellable Houses (HGTV),25,322500,7.93
3,Halloween Wars (Food Network),18,174600,2.91


Plan cost ($): 993100
Plan target impressions (M): 15.2
Blended target CPM ($): 65.42
Budget reaching non-heavy buyers (%): 77.9


### Task 5.2 Recommendation

**Answer**: I recommend spending 993,100 US dollars across 43 cable spots on Unsellable Houses (HGTV) and Halloween Wars (Food Network) plus 3 broadcast spots on Dancing with the Stars (ABC) and NCIS (CBS) to deliver 15.2M heavy athletic-shoe buyer impressions at a blended target CPM of $65.42. The cable shows deliver the lowest target CPMs (40.66 US dollars and 59.90 US dollars) and highest concentration indexes (127 and 116), while the two broadcast programs add immediate mass reach of over 1.2M target viewers per spot. We skipped expensive NFL telecasts because their target CPMs exceed 170 US dollars and live sports carry high schedule-shift risk. Delivery carries moderate uncertainty given our 38.6% historical MAPE for heavy buyers, meaning individual spots can under-deliver and require make-good spots (ADUs) if bought on guarantee. About 77.9% of the spend reaches adults outside the heavy-buyer segment based on program composition. However, much of this overspray still reaches light athletic-shoe buyers who purchase 1 to 2 pairs a year. After the November flight, we should audit as-run logs against panel and weighted ACR data to verify delivered target impressions, net reach, frequency distribution, and final eCPM.